In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from openpyxl import load_workbook
import datetime as dt
from openpyxl import Workbook
import seaborn as sns
import mysql.connector
from sqlalchemy import create_engine
import itertools
from scipy.signal import savgol_filter
from itertools import product

In [2]:
#----------------------CREAR DATAFRAME---------------------------#
def create_DF(dir,sheet,ncolumns,convert):
    """
        dir: directorio del archivo excel,
        sheet: nombre de la hoja,
        ncolumns: columnas a importar,
        convert: cambiar tipos de datos de columnas
    """
    df = pd.read_excel(
                       dir, 
                       sheet_name=sheet, 
                       na_values=['(en blanco)'],
                       header=0,usecols=ncolumns,
                       converters= convert
                      )
    df = pd.DataFrame(df)
    return df

def coloca_pico(edad):
    if edad<=35:
        return 1
    elif edad>=36 and edad<=72:
        return 2
    else:
        return 3
    
def edad_max(edad):
    if edad <= 35:
        return 35
    elif edad > 35 and edad <= 70:
        return 70
    else:
        return 102

In [3]:
#############################################################################################################
#directorios historico de siembras
dir_ = "//172.10.18.2/usuarios/Hilder/Data/tabla_curvas_clavel.xlsx"
#hoja
sh_ = "Hoja1"
#nombre de columnas
col_ = ['finca','bloque','nvariedad','cosecha','edad','f_m']
con_ = {'finca':str,'cosecha':str}

In [4]:
df_= create_DF(dir_,sh_,col_,con_)

In [5]:
df_ = df_.rename({'lunes':'fecha'}, axis=1)

In [6]:
df_['cosecha'] = df_['cosecha'].fillna('')  # Reemplaza los valores NaN con una cadena vacía
df_ = df_[df_['cosecha'].str.contains('25|23|24')]

In [7]:
#df_peaks= create_DF(dir_peaks,sh_peaks,col_peaks,con_peaks)

In [8]:
#df = pd.merge(df_,df_peaks, on='edad', how='left')
data = df_.copy()

In [9]:
data = data.query("edad > 15")

In [10]:
# Filtrar el DataFrame solo tercdfos
#df = df.loc[~((df['cosecha'].astype(str).str.contains('21|22|23')) & (df['pico'] < 3))]

In [11]:
data.head()

,nvariedad,finca,bloque,f_m,cosecha,edad
155,AMICO LAVANDER,001,23,0.0103,FEB24,22
156,AMICO LAVANDER,001,23,0.1076,FEB24,23
157,AMICO LAVANDER,001,23,0.2152,FEB24,24
158,AMICO LAVANDER,001,23,0.4489,FEB24,25
159,AMICO LAVANDER,001,23,0.5332,FEB24,26


## filtrar las que aplican y las que no

In [12]:
df_max_edad = data.groupby(['finca','bloque','nvariedad','cosecha'])['edad'].max().reset_index()

In [13]:
df_max_edad.head()

,finca,bloque,nvariedad,cosecha,edad
0,001,16,ANASTASIA,AGO24,63
1,001,16,ANASTASIA,EAS23,74
2,001,16,ANTIGUA,FEB25,55
3,001,16,APPLE TEA,NOV23,62
4,001,16,BERNARD,NOV23,62


In [14]:
df_max_edad['edad'] = df_max_edad.edad.apply(edad_max)

In [15]:
df_max_edad.head()

,finca,bloque,nvariedad,cosecha,edad
0,001,16,ANASTASIA,AGO24,70
1,001,16,ANASTASIA,EAS23,102
2,001,16,ANTIGUA,FEB25,70
3,001,16,APPLE TEA,NOV23,70
4,001,16,BERNARD,NOV23,70


In [16]:
df_nuevo = []

for i, row in df_max_edad.iterrows():
    #en cada iteraccion crea un nuevo dataframe con filas de edad desde 18 hasta la edad del df_max_edad,
    #luego guarda los resulados en df_nuevo y continua
    # Crea un nuevo DataFrame con filas desde 18 hasta la edad actual
    edad_actual = row['edad']
    nuevas_filas = pd.DataFrame({
        'finca': [row['finca']] * (edad_actual - 17),
        'bloque': [row['bloque']] * (edad_actual - 17),
        'nvariedad': [row['nvariedad']] * (edad_actual - 17),
        'cosecha': [row['cosecha']] * (edad_actual - 17),
        'edad': range(18, edad_actual + 1)
    })

    # Guarda el nuevo DataFrame en df_nuevo
    df_nuevo.append(nuevas_filas)

# Concatena todos los DataFrames en df_nuevo en un solo DataFrame
df_nuevo = pd.concat(df_nuevo, ignore_index=True)

In [17]:
df = pd.merge(
    df_nuevo,
    data,
    on=['finca','bloque','nvariedad','cosecha','edad'],
    how='left'
)

In [18]:
#df = df.fillna(0)

In [19]:
df.head()

,finca,bloque,nvariedad,cosecha,edad,f_m
0,001,16,ANASTASIA,AGO24,18,NaN
1,001,16,ANASTASIA,AGO24,19,NaN
2,001,16,ANASTASIA,AGO24,20,NaN
3,001,16,ANASTASIA,AGO24,21,NaN
4,001,16,ANASTASIA,AGO24,22,0.0166


In [20]:
df.to_excel("//svrinverpalmas/usuarios/Hilder/Data/tabla_curvas_clavel_zeros.xlsx", index=False)